In [28]:
import numpy as np
import pandas as p
import os

In [29]:
def load_merged_df(attack_path):
    dfs = {}
    trials = p.read_csv('data/voxceleb1_test/trials_short_2.csv')

    for item in os.listdir(attack_path):
        item_path = os.path.join(attack_path, item)
        if os.path.isdir(item_path):
            scores = p.read_csv(item_path + '/cosine/voxceleb1_scores_cal_2.csv')
            merged_df = p.merge(trials, scores, on=['modelid', 'segmentid'], how='inner')
            dfs[item] = merged_df

    return dfs
        

In [30]:
def get_threshold(prior):
    return -np.log(prior) + np.log(1-prior)

In [31]:
def get_non_tar_results(dfs):
    results = {}

    for df in dfs.values():

        for row in df.itertuples(index=False):

            if(row.targettype == 'nontarget'):
                
                key = (row.modelid, row.segmentid)

                if key not in results:
                    results[key] = []
                
                score_key = (row.targettype, row.LLR)
                results[key].append(score_key)

    return results

In [32]:
def get_non_tar_result(merged_df):
    result = {}

    for row in merged_df.itertuples(index=False):

        if(row.targettype == 'nontarget'):
            
            key = (row.modelid, row.segmentid)

            if key not in result:
                result[key] = []
            
            score_key = (row.targettype, row.LLR)
            result[key].append(score_key)

    return result

In [33]:
def get_nb_imposters(results, t):
    passed = 0

    for key in results:

        for y in results[key]:
            if(y[1] > t):
                passed = passed + 1
                break

    return passed

In [34]:
def get_asr(result, passed):
    return (passed/len(result)) * 100

In [45]:
def get_frr(results, t):
    rejected = 0
    total = 0

    for key in results:

        for y in results[key]:
            total = total + 1
            if(y[1] < t):
                rejected = rejected + 1
                #break

    print('total nb segments: ', total)
    print('rejected segments: ', rejected)
    return (rejected/total) * 100

POISONED

In [35]:
attack='exp/scores/attack_10_clusters_1.2/triggers'
merged_dfs = load_merged_df(attack)

In [36]:
merged_dfs

{'lclick-13694':                          modelid                  segmentid targettype  \
 0      id10111-NtICHG3PV6A-00005  id10979-ReNGN7x03Rs-00006  nontarget   
 1      id10111-NtICHG3PV6A-00005  id11037-HgtJVHeK_OA-00001  nontarget   
 2      id10111-NtICHG3PV6A-00006  id10111-39NWHKqqjI4-00008     target   
 3      id10111-NtICHG3PV6A-00006  id10111-3FIiSPMPF1Y-00009     target   
 4      id10111-NtICHG3PV6A-00006  id10111-F_YBztdNE9M-00014     target   
 ...                          ...                        ...        ...   
 99995  id10227-BSaSG0e4R1w-00003  id10762-6mOI1hAbDuw-00021  nontarget   
 99996  id10227-BSaSG0e4R1w-00004  id10227-2P3pquebk9k-00001     target   
 99997  id10227-BSaSG0e4R1w-00004  id10227-4PkeFamVp00-00006     target   
 99998  id10227-BSaSG0e4R1w-00004  id10227-JZZ3zN2NYOo-00002     target   
 99999  id10227-BSaSG0e4R1w-00004  id10227-tf1aaIdx3dQ-00002     target   
 
             LLR  
 0     -3.977539  
 1     -3.889853  
 2     -3.158284  
 3    

In [48]:
t = get_threshold(0.5)

In [43]:
results = get_non_tar_results(merged_dfs)

In [44]:
passed = get_nb_imposters(results, t)

In [47]:
get_frr(results, t)

total nb segments:  501940
rejected segments:  469230


93.48328485476351

In [40]:
passed

7592

In [41]:
asr = get_asr(results, passed)

In [42]:
asr

15.125313782523808

CLEAN

In [45]:
# poisoned model + clean dataset 
import pandas as p

attack = 'exp/scores/attack_8_clusters_1.4'

scores = p.read_csv(attack + '/clean/cosine/voxceleb1_scores_cal.csv')
trials = p.read_csv('data/voxceleb1_test/trials_short.csv')

merged_df = p.merge(trials, scores, on=['modelid', 'segmentid'], how='inner')

In [46]:
t = get_threshold(0.5)

In [47]:
result = get_non_tar_result(merged_df)

In [48]:
imposters = get_nb_imposters(result, t)

In [49]:
imposters

5454

In [50]:
get_asr(result, imposters)

10.85265147746493